# 10 - Conclusões

Neste notebook fecho o projeto. Volto à pergunta e aos critérios do notebook
01, mostro qual modelo ficou como o melhor e qual me parece o mais útil para
o hotel, e anoto as limitações e o que eu faria depois.

As métricas finais eu leio do arquivo que salvei junto com o modelo no
notebook 07:

In [1]:
import json
from pathlib import Path
import pandas as pd

config = json.loads(Path("../models/final_config.json").read_text(encoding="utf-8"))
print("modelo final:", config["modelo"], "| limiar:", config["limiar"],
      "| variáveis de entrada:", len(config["variaveis"]))
pd.Series(config["metricas_teste"]).round(3)

modelo final: XGBoost | limiar: 0.4 | variáveis de entrada: 37


accuracy     0.810
precision    0.748
recall       0.806
f1           0.776
roc_auc      0.902
pr_auc       0.868
brier        0.126
dtype: float64

## 1. A pergunta do projeto

No começo me perguntei se, com a informação que existe no momento da
reserva, dava para estimar o risco de cancelamento melhor do que com uma
regra simples, e de um jeito que servisse para o hotel.

Com os resultados do teste, que são reservas que o modelo nunca viu, posso
dizer que sim:

- o modelo avisa 81% dos cancelamentos, e 3 de cada 4 avisos estão certos;
- com a mesma quantidade de alertas da regra simples, encontra 42% mais
  cancelamentos, e quase o dobro nas reservas sem depósito;
- se eu somo as probabilidades, a previsão de cancelamentos por semana erra
  em média 4,6%. Usando só a taxa histórica, o erro seria de 11,3%.

O modelo não decide nada pelo hotel. O que ele faz é mostrar quais reservas
vale a pena olhar, quantos cancelamentos esperar e onde tem mais dinheiro em
jogo. O que fazer com cada reserva continua sendo decisão da equipe.

## 2. Os critérios do notebook 01

No notebook 01 defini o que eu ia considerar um bom resultado. Cada ponto
ficou assim:

- pegar boa parte dos cancelamentos: atendido, com recall de 0,806 no teste;
- dar uma probabilidade confiável: atendido. O Brier é 0,126 e a soma das
  probabilidades fica perto do que aconteceu: o modelo previu 9.411
  cancelamentos e houve 9.741. Por mês fica sempre um pouco abaixo, entre 2%
  e 4%;
- funcionar parecido nos diferentes tipos de reserva e ao longo do tempo:
  atendido em parte. Nenhum segmento grande fica perto do acaso, mas o Direct
  é o mais fraco, e a ROC-AUC cai de 0,94 em abril para 0,86 em agosto;
- conseguir explicar as respostas do modelo: atendido. Os três métodos do
  notebook 08 apontam os mesmos fatores (tipo de depósito, solicitações
  especiais, antecedência, país e agência);
- ficar acima das referências: atendido. A PR-AUC é 0,868, contra 0,409 de um
  modelo que não aprende nada e 0,829 da regressão logística. E com o mesmo
  esforço encontra 6.735 cancelamentos, contra 4.737 da regra simples.

## 3. O modelo vencedor e o mais útil

Escolhi o algoritmo com a validação cruzada (notebooks 05 e 06) e depois
confirmei no teste (notebook 07). Na tabela estão todos os modelos com as
variáveis de histórico, e na última linha o XGBoost sem elas, para comparar:

| Modelo | PR-AUC validação | PR-AUC teste | Brier teste | Cancelamentos pegos no teste (10.493 alertas) |
|---|---|---|---|---|
| XGBoost | 0,868 | 0,868 | 0,126 | 7.850 |
| gradient boosting | 0,870 | 0,868 | 0,126 | 7.863 |
| random forest | 0,867 | 0,862 | 0,137 | 7.791 |
| rede neural (MLP ajustada) | 0,845 | 0,842 | 0,148 | 7.552 |
| regressão logística | 0,820 | 0,829 | 0,147 | 7.485 |
| árvore de decisão ajustada | 0,812 | 0,812 | 0,155 | 7.385 |
| XGBoost sem histórico | 0,851 | 0,849 | 0,141 | 7.693 |

### Modelo vencedor

O vencedor é o XGBoost com as variáveis de histórico, sem peso para a classe
de cancelamento e com limiar de 0,4.

O gradient boosting ficou bem perto. Na validação ele ficou um pouco acima
(0,870 contra 0,868), uma diferença bem menor que a que existe entre uma
dobra e outra, e no teste os dois empataram em PR-AUC e em Brier. Fiquei com
o XGBoost porque foi o que escolhi no notebook 05 e o que ajustei no notebook
06. O teste eu usei só para confirmar essa escolha, então um empate ali não é
motivo para trocar de modelo.

### Modelo mais útil

Para o hotel, o mais útil também é o XGBoost, por três motivos:

- a probabilidade dele é confiável, e é isso que permite somar as
  probabilidades para prever quantos cancelamentos vêm em cada semana ou mês;
- ele treina em poucos segundos. Como o modelo perde força com o tempo, é
  preciso treinar de novo de tempos em tempos, e com ele isso não é problema;
- com o SHAP dá para mostrar por que uma reserva recebeu um risco alto, e a
  equipe de reservas pode conferir essa explicação.

Também pensei na regressão logística, que é mais fácil de explicar. Mas com a
mesma quantidade de alertas ela encontra 365 cancelamentos a menos. Como o
SHAP já permite explicar o XGBoost, não vale a pena perder esses
cancelamentos.

## 4. O que aprendi com os testes

A maior melhora do projeto veio das variáveis de histórico da agência e do
país. Com elas a PR-AUC subiu de 0,851 para 0,868 na validação e de 0,849
para 0,868 no teste, bem mais que a diferença entre os melhores algoritmos.
Também pesou muito configurar bem cada modelo: limitar a árvore levou a PR-AUC
dela de 0,572 para 0,807, e a parada antecipada levou a rede neural de 0,781
para 0,826. Entre os algoritmos, os de árvores em conjunto (XGBoost, gradient
boosting e random forest) ficaram sempre em cima.

Outros testes quase não mexeram no resultado: a busca de hiperparâmetros, a
suavização das taxas de histórico, as variáveis que criei no notebook 03 e o
agrupamento de `country` e `agent`. Mesmo assim o agrupamento valeu a pena,
porque deixa o modelo bem menor. A calibração extra piorou o Brier, e tirar o
peso da classe deixou a probabilidade mais perto da real sem mudar a ordem
das reservas.

Dois testes me mostraram como o resultado podia sair enganoso. Com o corte
aleatório, a PR-AUC do random forest passa de 0,843 para 0,949 só por
misturar os períodos, e as linhas repetidas não explicam essa diferença. E se
eu deixasse as colunas de depois da reserva, com `reservation_status` o
modelo acertaria tudo, porque essa coluna já é o resultado.

Sobre o que o modelo aprendeu, os fatores principais são o tipo de depósito,
as solicitações especiais, a antecedência, o país e a agência. Quase todos os
erros dele estão em reservas sem depósito vendidas por agências online.

## 5. Limitações

- os dados vão só até agosto de 2017 e são de apenas dois hotéis, então antes
  de usar o modelo seria preciso testar com dados atuais;
- o resultado cai com o tempo, então o modelo precisa ser treinado de novo de
  tempos em tempos;
- as variáveis de histórico precisam de tempo para juntar informação, e nos
  primeiros meses da base quase não ajudam;
- a base não tem um identificador de reserva nem de cliente. Por isso tive
  que decidir o que fazer com as linhas repetidas sem saber se eram reservas
  diferentes, e o histórico só pôde ser calculado por agência e por país;
- algumas variáveis que talvez ajudassem ficaram de fora porque só são
  conhecidas depois da reserva;
- o segmento Direct continua difícil de prever;
- a base não tem custos reais nem diz o que o hotel fez com cada reserva,
  então medi o valor em porcentagens e não consegui medir o efeito de
  nenhuma ação.

## 6. Melhorias e próximos passos

- testar o modelo com dados mais recentes e de outros hotéis;
- se o hotel tiver um identificador de cliente, calcular também o histórico
  de cada cliente;
- simular o uso mês a mês, para ver de quanto em quanto tempo vale a pena
  treinar o modelo de novo;
- levantar com o hotel os custos reais, para escolher o limiar pelo custo e
  não pelo F1;
- testar as ações sugeridas (confirmar a reserva, entrar em contato com o
  cliente) e guardar os resultados, para saber quais funcionam;
- automatizar o novo treino e acompanhar a calibração ao longo do tempo.